In [ ]:
import asyncio
import os
from dotenv import load_dotenv
from google.adk import Agent
from google.adk.runners import InMemoryRunner
from google.genai import types
from datetime import datetime, timedelta

Load environment variables from .env file

In [ ]:
load_dotenv()

 Tool callbacks

In [ ]:
def get_date(x_days_from_today:int):
    """
    Retrieves a date for today or a day relative to today.
    Args:
        x_days_from_today (int): how many days from today? (use 0 for today)
    Returns:
        A dict with the date in a formal writing format. For example:
        {"date": "Wednesday, May 7, 2025"}
    """
    target_date = datetime.today() + timedelta(days=x_days_from_today)
    date_string = target_date.strftime("%A, %B %d, %Y")
    return {"date": date_string}

Basic configuration<br>
Ensure you have the GOOGLE_API_KEY environment variable set.

In [ ]:
DEFAULT_MODEL = "gemini-flash-latest"

In [ ]:
model_name = os.getenv("MODEL", DEFAULT_MODEL)

1. Initialize the LLM Agent<br>
We define the model and the persona (instruction)<br>
This must be exposed as 'root_agent' for the adk CLI to find it.

In [ ]:
root_agent = Agent(
    model=model_name,
    name="helpful_assistant",
    instruction="You are a helpful and friendly AI assistant. Keep answers concise.",
    tools=[get_date]
)

Force using Google AI API (Local/Non-Vertex)<br>
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "0"

In [ ]:
async def main():
    api_key = os.getenv("GOOGLE_API_KEY")
    if not api_key:
        print("Error: GOOGLE_API_KEY not found in environment variables.")
        return
    
    print(f"Initializing agent with model: {model_name} (Local/API Key mode)")

    # 2. Set up the Runner
    # The runner handles the orchestration of the agent and state.
    # InMemoryRunner keeps the session state in memory.
    runner = InMemoryRunner(
        agent=root_agent,
        app_name="my_adk_app",
    )

    # 3. Create a Conversation Session
    # Sessions track the history of the conversation for a user.
    user_id = "local_user"
    session = await runner.session_service.create_session(
        app_name="my_adk_app",
        user_id=user_id
    )
    print(f"Session created (ID: {session.id})")
    print("Type 'exit' or 'quit' to stop.")

    # 4. Run the Conversation Loop
    while True:
        try:
            user_input = input("\nUser: ").strip()
        except EOFError:
            break
        if not user_input or user_input.lower() in ["exit", "quit"]:
            print("Goodbye!")
            break

        # Construct the message object using google.genai types
        content = types.Content(
            role="user",
            parts=[types.Part.from_text(text=user_input)]
        )
        print("Agent: ", end="", flush=True)
        
        # runner.run_async executes the turn and yields events (e.g. chunks of response)
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=content,
        ):
            # Process and display the response content
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        print(part.text, end="", flush=True)
        print() # New line after response

In [ ]:
if __name__ == "__main__":
    asyncio.run(main())